# AI/BI Dashboards & Genie

**Module 7 | Fundamentals Training BONUS**

The Gold layer data is ready. Now it needs to be presented to the business. Databricks offers two tools:
- **AI/BI Dashboards** — interactive dashboards built on SQL, with no need to export to external BI tools
- **AI/BI Genie** — "conversation" with data in natural language — business users ask questions in plain English and receive charts

## Topics

| # | Topic |
|---|-------|
| 1 | SQL Editor — direct queries |
| 2 | AI/BI Dashboards — creating a dashboard |
| 3 | AI/BI Genie — natural language analytics |
| 4 | End-to-end demo: Gold table → Dashboard → Genie |


## Learning Objectives

After completing this module you will be able to:

- **Use** the SQL Editor in Databricks for ad-hoc queries
- **Create** a simple AI/BI Dashboard with charts and filters
- **Publish** a dashboard for other users
- **Configure** a Genie Space and ask a question in natural language
- **Know** when to use a Dashboard and when to use Genie


## Setup

In [0]:
%run ../setup/00_setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

print(f"Catalog: {CATALOG}")
print(f"Gold:    {CATALOG}.{GOLD_SCHEMA}")

---
## Preparing Demo Data

If you do not yet have data in the Gold layer (the Final Project has not been completed), let's quickly create a demo table.


In [0]:
# Check whether the Gold layer has any data
gold_tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}").count()
print(f"Tables in Gold: {gold_tables}")

if gold_tables > 0:
    display(spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}"))
else:
    print("Gold layer is empty — creating demo data in the next cell.")


In [0]:
# Create realistic demo data for RetailHub
# Even if you already have data from the Final Project, this table is useful for the AI/BI demo

demo_sales_table = f"{CATALOG}.{GOLD_SCHEMA}.demo_sales_summary"

spark.sql(f"""
    CREATE OR REPLACE TABLE {demo_sales_table}
    USING DELTA
    COMMENT 'Aggregated RetailHub sales data — AI/BI demo'
    AS
    SELECT
        col1  AS order_month,
        col2  AS region,
        col3  AS product_category,
        col4  AS total_orders,
        col5  AS total_revenue,
        col6  AS avg_order_value,
        col7  AS unique_customers
    FROM VALUES
        ('2024-01', 'North', 'Electronics', 142, 28400.0, 200.0, 98),
        ('2024-01', 'South', 'Electronics', 98,  19600.0, 200.0, 67),
        ('2024-01', 'North', 'Clothing',    234, 11700.0,  50.0, 198),
        ('2024-01', 'South', 'Clothing',    187,  9350.0,  50.0, 160),
        ('2024-01', 'East',  'Food',        312,  9360.0,  30.0, 290),
        ('2024-02', 'North', 'Electronics', 165, 33000.0, 200.0, 112),
        ('2024-02', 'South', 'Electronics', 112, 22400.0, 200.0, 78),
        ('2024-02', 'North', 'Clothing',    256, 12800.0,  50.0, 220),
        ('2024-02', 'South', 'Clothing',    201, 10050.0,  50.0, 174),
        ('2024-02', 'East',  'Food',        289,  8670.0,  30.0, 267),
        ('2024-03', 'North', 'Electronics', 198, 39600.0, 200.0, 134),
        ('2024-03', 'South', 'Electronics', 134, 26800.0, 200.0, 91),
        ('2024-03', 'North', 'Clothing',    289, 14450.0,  50.0, 245),
        ('2024-03', 'South', 'Clothing',    223, 11150.0,  50.0, 190),
        ('2024-03', 'East',  'Food',        401, 12030.0,  30.0, 370),
        ('2024-04', 'North', 'Electronics', 178, 35600.0, 200.0, 120),
        ('2024-04', 'South', 'Electronics', 145, 29000.0, 200.0, 99),
        ('2024-04', 'North', 'Clothing',    310, 15500.0,  50.0, 265),
        ('2024-04', 'South', 'Clothing',    241, 12050.0,  50.0, 207),
        ('2024-04', 'East',  'Food',        378, 11340.0,  30.0, 348)
"""
)
print(f"Table '{demo_sales_table}' is ready.")
display(spark.table(demo_sales_table))


In [0]:
# Top customers table
demo_customers_table = f"{CATALOG}.{GOLD_SCHEMA}.demo_top_customers"

spark.sql(f"""
    CREATE OR REPLACE TABLE {demo_customers_table}
    USING DELTA
    COMMENT 'Top RetailHub customers by order value — AI/BI demo'
    AS
    SELECT col1 AS customer_id, col2 AS full_name, col3 AS city,
           col4 AS total_orders, col5 AS total_spent, col6 AS segment
    FROM VALUES
        (1001, 'Anna Collins',    'Warsaw',  45, 8900.0,  'Premium'),
        (1002, 'Peter Adams',      'Krakow',  38, 7600.0,  'Premium'),
        (1003, 'Martha Spencer', 'Gdansk',  29, 5800.0,  'Regular'),
        (1004, 'Jacob Grant',      'Warsaw',  52, 10400.0, 'VIP'),
        (1005, 'Caroline Blake',   'Wroclaw', 33, 6600.0,  'Premium'),
        (1006, 'Tomasz Lewandowski','Warsaw', 18, 3600.0,  'Regular'),
        (1007, 'Agatha Stone',   'Lodz',   41, 8200.0,  'Premium'),
        (1008, 'Marcus Wright',    'Poznan',  61, 12200.0, 'VIP'),
        (1009, 'Ewa Lis',          'Krakow',  22, 4400.0,  'Regular'),
        (1010, 'Krzysztof Baran',  'Warsaw',  57, 11400.0, 'VIP')
""")
print(f"Table '{demo_customers_table}' is ready.")


---
## 1. SQL Editor

The SQL Editor is a **lightweight SQL interface** in Databricks — an alternative to notebooks for analysts and BI specialists who work primarily in SQL.

### How to open it:
1. Click **SQL Editor** in the left menu (or **New → SQL Editor**)
2. In the top-right corner select a **SQL Warehouse** (not a notebook cluster!)
3. Start writing SQL

![image_1772655195403.png](assets/images/image_1772655195403.png "image_1772655195403.png")

### Try the following queries in the SQL Editor (not in this notebook):


In [0]:
# Paste these queries into the SQL Editor in the GUI
# (running them here also works, but the goal is to get familiar with the interface)

print("Query 1 — Revenue trend by month:")
print(f"""
SELECT
    order_month,
    SUM(total_revenue) AS monthly_revenue,
    SUM(total_orders)  AS monthly_orders
FROM {demo_sales_table}
GROUP BY order_month
ORDER BY order_month;
""")

print("Query 2 — Revenue and orders by region:")
print(f"""
SELECT
    region,
    SUM(total_revenue) AS total_revenue,
    SUM(total_orders)  AS total_orders,
    ROUND(AVG(avg_order_value), 2) AS avg_order_value
FROM {demo_sales_table}
GROUP BY region
ORDER BY total_revenue DESC;
""")


In [0]:
# Run in notebook — preview data before creating the dashboard
display(
    spark.sql(f"""
        SELECT order_month, region,
               SUM(total_revenue) AS revenue,
               SUM(total_orders) AS orders
        FROM {demo_sales_table}
        GROUP BY order_month, region
        ORDER BY order_month, revenue DESC
    """)
)


---
## 2. AI/BI Dashboards — step-by-step creation

AI/BI Dashboards replaced the old Databricks SQL Dashboards in 2024. They are faster, more polished, and support AI-assisted authoring.

### Step 1 — Create a new Dashboard
1. Left menu: **New** → **Dashboard** (or Workspace → Create → Dashboard)
2. Give it a name: `RetailHub Sales Overview — [your username]`

### Step 2 — Add the first dataset (Revenue Trend)
1. Click **Add dataset** → **Create from SQL**
2. In the SQL field enter:

`sql
SELECT order_month, SUM(total_revenue) AS revenue
FROM demo_sales_summary
GROUP BY order_month
ORDER BY order_month`


3. Name the dataset: `Monthly Revenue`

### Step 3 — Add a visualisation based on the dataset
1. Click **Add visualization** on the `Monthly Revenue` dataset
2. Set the chart type: **Line Chart**
3. X axis: `order_month`, Y axis: `revenue`
4. Title: `Monthly Revenue Trend`

### Step 4 — Add a dataset for Revenue by Region
1. Click **Add dataset** → **Create from SQL**
2. SQL:

`sql
SELECT region, product_category, SUM(total_revenue) AS revenue
FROM demo_sales_summary
GROUP BY region, product_category
ORDER BY revenue DESC`


3. Name the dataset: `Revenue by Region & Category`

### Step 5 — Add a bar chart visualisation
1. Click **Add visualization** on the `Revenue by Region & Category` dataset
2. Type: **Bar Chart**, Stacked
3. X: `region`, Y: `revenue`, Color: `product_category`
4. Title: `Revenue by Region & Category`

### Step 6 — Add a KPI tile
1. Click **Add dataset** → **Create from SQL**
2. SQL: 

sql
SELECT SUM(total_revenue) AS total_revenue FROM demo_sales_summary


3. Name the dataset: `Total Revenue`
4. Click **Add visualization** on the `Total Revenue` dataset
5. Type: **Counter** (or Metric)
6. Title: `Total Revenue`, Format: `$#,##0`

### Step 7 — Add a filter (date picker)
1. **Add widget** → **Filter** → Drop-down
2. Title: `Month`, field: bind to `order_month` from your datasets/visualisations

### Step 8 — Publish the dashboard
1. Click **Publish** (top-right corner)
2. Choose **Share** → add a colleague as viewer or editor
3. Tick **Schedule** → set refresh every 24 h

---
## 3. AI/BI Genie — Natural Language Analytics

Genie is a space ("Genie Space") where a business user can **ask questions about data in natural language** and receive ready-made charts or answers.

No SQL knowledge is required. Genie builds and executes the query automatically.

### Creating a Genie Space
1. Left menu: **New** → **Genie Space**
2. Name: `RetailHub Analytics — [your username]`
3. **Add tables** → add:
   - `{CATALOG}.gold.demo_sales_summary`  
   - `{CATALOG}.gold.demo_top_customers`
4. (Optional) Add **context instructions** — e.g.:
   > `total_revenue is always in USD. Segment 'VIP' means customers spending over 10,000 USD per year.`
5. Click **Save**

### Questions to test

Type each question into the Genie input field and watch how it generates a query and a chart:


In [0]:
# Sample questions you can type into Genie (these are strings, not code!)
genie_questions = [
    "Which month had the highest revenue?",
    "Show me the sales trend for Electronics in the North region",
    "How many VIP customers do we have?",
    "Which city generates the most revenue?",
    "Compare the North and South regions by number of orders",
    "What is the average spend per customer in each city?",
    "In which month did we record the largest month-over-month growth?",
]

print("Questions to test in the Genie Space:")
for i, q in enumerate(genie_questions, 1):
    print(f"  {i}. {q}")


### When Genie works best:
- Aggregation questions (`sum`, `avg`, `count`, `top N`)
- Comparisons (`North vs South`, `this month vs last month`)
- Time-based trends (`month over month`, `growth`)
- Filtering by value (`Electronics only`, `VIP customers only`)

### When Genie has limitations:
- Complex joins across many tables without good context
- Business calculations requiring additional logic (e.g., LTV, attribution)
- Queries that depend on precise knowledge of poorly named columns (e.g., `col_a_v2_final`)

> **Trainer tip:** A good column name = `total_revenue_usd`, a bad one = `c1`. Genie is only as good as the column names and context instructions.


---
## 4. When to use which tool — decision table

| Tool | For whom | When | Limitations |
|------|----------|------|-------------|
| **Notebook** | Data Engineers, Data Scientists | ELT, pipelines, ML, exploration | Requires Python / SQL knowledge |
| **SQL Editor** | SQL Analysts | Ad-hoc queries, debugging, preparing SQL for dashboards | No visualisation |
| **AI/BI Dashboard** | Analysts, BI, management | Regular reports, KPIs, monitoring | Static definition — questions are predictable |
| **AI/BI Genie** | Business Users, Analysts | Deep-dive, ad-hoc questions without SQL | Limited to data in the Genie Space |

> **Typical company flow:**  
> Data Engineer (Notebook) → writes to Gold Delta → Analyst (SQL Editor) → validates → AI/BI Dashboard → management views KPIs daily → Business User (Genie) → asks their own questions


In [0]:
# display() in a notebook — how it works and how to create a chart
# ==============================================================
# After running this cell:
# 1. The result appears as a TABLE
# 2. Click the "+" icon next to "Table" (top-right of the result widget)
# 3. Choose chart type: Bar Chart
# 4. X axis: product_category, Y axis: revenue
# 5. Click "Save" — the chart will remain in the notebook

display(
    spark.sql(f"""
        SELECT 
            product_category,
            SUM(total_revenue) AS revenue,
            SUM(total_orders)  AS orders
        FROM {demo_sales_table}
        GROUP BY product_category
        ORDER BY revenue DESC
    """)
)


In [0]:
# Second example — revenue trend by month
# Click "+" → Line Chart → X: order_month, Y: revenue

display(
    spark.sql(f"""
        SELECT
            order_month,
            region,
            SUM(total_revenue) AS revenue
        FROM {demo_sales_table}
        GROUP BY order_month, region
        ORDER BY order_month
    """)
)

---
## Summary

### What you accomplished:
- Gold layer data is now **accessible to the business** without exporting
- SQL Editor → fast queries without a notebook cluster
- AI/BI Dashboard → regular report with KPIs, charts, and a filter
- AI/BI Genie → business user asks in plain English and gets a chart

### Key facts:
1. AI/BI Dashboards use a **SQL Warehouse** — separate, dedicated compute for BI
2. Dashboards can be **refreshed automatically** (scheduled) and **emailed**
3. Genie works best when tables have **descriptive column names** and context instructions
4. Everything is **managed by Unity Catalog** — the same permissions as in notebooks

> **Next steps — Databricks beyond the fundamentals:**  
> Lakeflow Pipelines (declarative pipelines), Delta Sharing, Row/Column Level Security, MLflow basics
